# Cleaning Netflix Dataset 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
netflix = spark.read.format("csv")\
         .option("header", "true")\
        .option("multiLine", "true")\
         .option("quote", '"') \
        .option("escape", '"') \
        .option("mode", "PERMISSIVE")\
        .option("inferSchema", "true")\
        .load("/Volumes/workspace/default/day-2-dataset/netflix_titles.csv")

In [0]:
netflix.printSchema()
netflix.show(5)

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)

+-------+-------+-----+-----------------+--------------------+-------------+-----------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|title|         director|                cast|      country|       date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+-----+-----------------+--------------------+-------------+-----------------+------------+------+---------+--------------------+--------------------+
|     s1|TV Show|

## Data Exploration 

In [0]:
print("Total no of rows: ", netflix.count())
print("Total no of columns: ", len(netflix.columns))
print("unique rows: ", netflix.distinct().count())

Total no of rows:  7787
Total no of columns:  12
unique rows:  7787


total rows and unique rows are same so no duplicates

## Standardising the column name 

Nthong to standerdise , the column names are already in standerd format


## Handling missing values

In [0]:
#checking for null values 
for c in netflix.columns:
    print(c, netflix.filter(col(c).isNull()).count())

show_id 0
type 0
title 0
director 2389
cast 718
country 507
date_added 10
release_year 0
rating 7
duration 0
listed_in 0
description 0


In [0]:
netflix = netflix.na.drop(subset=["date_added", "rating"])
netflix = netflix.na.fill("Unknown", subset=["director", "cast", "country"])

print("After dropping null: ", netflix.count())

print("After handling nulls:")
for c in netflix.columns:
    print(c, netflix.filter(col(c).isNull()).count())


After dropping null:  7770
After handling nulls:
show_id 0
type 0
title 0
director 0
cast 0
country 0
date_added 0
release_year 0
rating 0
duration 0
listed_in 0
description 0


# Handling column by column 

In [0]:
print("unique column values: ")
for c in netflix.columns:
    print(c, netflix.select(c).distinct().count())

unique column values: 
show_id 7777
type 2
title 7777
director 4050
cast 6822
country 682
date_added 1565
release_year 73
rating 15
duration 216
listed_in 491
description 7759


In [0]:
#country, date_added, release_year, rating, duration, listed_in

netflix.select("country").distinct().show()

netflix.select("date_added").distinct().show()

netflix.select("release_year").distinct().show()

netflix.select("rating").distinct().show()

netflix.select("duration").distinct().show()

netflix.select("listed_in").distinct().show()

#Nothing to be cleaned 

+--------------------+
|             country|
+--------------------+
|         South Korea|
|           Indonesia|
|           Australia|
|United Kingdom, U...|
|Mauritius, South ...|
|      Spain, Belgium|
|United Kingdom, C...|
|               Ghana|
|Singapore, United...|
|India, United Kin...|
|       France, Qatar|
|              Israel|
|Ireland, United K...|
|United Kingdom, G...|
|     Spain, Portugal|
|Ireland, United S...|
|Germany, United S...|
|Netherlands, Germ...|
|              Turkey|
|            Thailand|
+--------------------+
only showing top 20 rows
+------------------+
|        date_added|
+------------------+
| November 30, 2018|
|      May 16, 2018|
|September 15, 2018|
|   October 2, 2020|
|   August 23, 2019|
|  January 26, 2017|
|     April 6, 2018|
|    April 14, 2020|
| December 28, 2020|
|    March 12, 2019|
| November 15, 2020|
|  October 31, 2020|
| December 30, 2018|
|  October 25, 2020|
|  October 13, 2017|
|   January 9, 2018|
|     April 5, 2019|
|  

# converting datatypes 

In [0]:
netflix = netflix.withColumn("release_year", col("release_year").cast("int"))

netflix.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = false)
 |-- cast: string (nullable = false)
 |-- country: string (nullable = false)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



# dropping unwanted columns

In [0]:
netflix = netflix.drop("show_id")

# data transformation 

In [0]:
#Print all column names using
netflix.columns

['type',
 'title',
 'director',
 'cast',
 'country',
 'date_added',
 'release_year',
 'content_rating',
 'duration',
 'listed_in',
 'description',
 'platform']

In [0]:
#Count total rows using
print("Total rows: ", netflix.count())

Total rows:  7770


In [0]:
#Display schema structure using
netflix.printSchema()

root
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = false)
 |-- cast: string (nullable = false)
 |-- country: string (nullable = false)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- content_rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)
 |-- platform: string (nullable = false)



In [0]:
#Print the number of columns using
print("Total columns: ", len(netflix.columns))

Total columns:  12


# No corrupt records 
there is no extra record detected like "_corrupt_record"

# Final Check

In [0]:
netflix.printSchema()
netflix.show(2)
print("Final rows: ", netflix.count())

print("Final check for nulls:")
for c in netflix.columns:
    print(c, netflix.filter(col(c).isNull()).count())

root
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = false)
 |-- cast: string (nullable = false)
 |-- country: string (nullable = false)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- content_rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)
 |-- platform: string (nullable = false)

+-------+-----+-----------------+--------------------+-------+-----------------+------------+--------------+---------+--------------------+--------------------+--------+
|   type|title|         director|                cast|country|       date_added|release_year|content_rating| duration|           listed_in|         description|platform|
+-------+-----+-----------------+--------------------+-------+-----------------+------------+--------------+---------+--------------------+--------------------+-----

# Storing the cleaned the dataset

In [0]:
netflix.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/default/day-2-dataset/netflix_cleaned")